# Deploy the compileml scorer with SageMaker SDK v3 `ModelBuilder`

Notebook version of `deploy_modelbuilder.py`, adapted to this project and to running **offline**:

| `deploy_modelbuilder.py` | this notebook |
| --- | --- |
| pulls a registered pyfunc from the MLflow App registry | uses the compileml artifact `artifacts/insurance_fraud.json` on disk |
| `model_metadata={"MLFLOW_MODEL_PATH": ...}` | `inference_spec=CompileMLSpec(...)`: v3's hook for a model that is not a torch / xgboost / tensorflow / sklearn object (`model=` insists on one of those, even with `image_uri`) |
| DLC or BYOC via `IMAGE_URI` | always our BYOC image `sagemaker-ai/container`, local tag `compileml-scorer:local` |
| `predictor.predict()` in local mode | `LocalEndpoint.invoke()` **and** `sagemaker.core.resources.Endpoint.invoke()` |

**Offline:** even in local mode the SDK makes three control-plane calls: it validates the execution role with IAM `SimulatePrincipalPolicy` inside `build()`, resolves the default S3 bucket through STS, and `docker pull`s the image. `sagemaker_offline.use_local_stubs()` (applied when `SM_OFFLINE=1`, the default here) replaces those three with local no-ops; everything else in the SDK runs as-is. Docker must be running and `compileml-scorer:local` must exist (`make image`, built once while online).

`MB_MODE=endpoint` deploys a serverless endpoint (last section, not executed here).

In [1]:
import csv
import json
import os
import shutil
import sys
import tempfile
from pathlib import Path

from sagemaker.serve.builder.schema_builder import SchemaBuilder
from sagemaker.serve.mode.function_pointers import Mode
from sagemaker.serve.model_builder import ModelBuilder
from sagemaker.serve.spec.inference_spec import InferenceSpec
from sagemaker.serve.utils.types import ModelServer

ROOT = Path.cwd().resolve()
if not (ROOT / "artifacts").exists():  # notebook started from sagemaker-ai/
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "sagemaker-ai"))  # for sagemaker_offline

# .env holds AWS_PROFILE / AWS_DEFAULT_REGION; only the region is used in local mode
for line in (
    (ROOT / ".env").read_text().splitlines() if (ROOT / ".env").exists() else []
):
    if "=" in line and not line.lstrip().startswith("#"):
        k, v = line.split("=", 1)
        os.environ.setdefault(k.strip(), v.strip())
os.environ.setdefault("AWS_DEFAULT_REGION", "us-east-1")

MB_MODE = os.environ.get("MB_MODE", "local")  # "endpoint" -> SAGEMAKER_ENDPOINT
MODEL_SERVER = ModelServer[
    os.environ.get("MODEL_SERVER", "TORCHSERVE")
]  # TORCHSERVE or MMS (see Part 1)
SM_OFFLINE = os.environ.get("SM_OFFLINE", "1") == "1" and MB_MODE == "local"
ACCOUNT = "823613469927"
IMAGE_URI = os.environ.get("IMAGE_URI", "compileml-scorer:local")
# MODEL_SERVER=SMD IMAGE_URI=public.ecr.aws/sagemaker/sagemaker-distribution:3.2.0-cpu runs the real
# SageMaker Distribution image; its Tornado server needs code/inference.py (see the spec cell).
IS_SMD_IMAGE = "sagemaker-distribution" in IMAGE_URI
# Validated by the SDK (IAM SimulatePrincipalPolicy) unless SM_OFFLINE stubs that out.
ROLE_ARN = os.environ.get(
    "SAGEMAKER_ROLE_ARN",
    f"arn:aws:iam::{ACCOUNT}:role/service-role/AmazonSageMaker-ExecutionRole-20260529T194818",
)
ARTIFACT = ROOT / "artifacts" / "insurance_fraud.json"
INPUT_CSV = ROOT / "data" / "Insurance_FraudulentAutoInsuranceClaims_2K_coldstart.csv"
ENDPOINT_NAME = os.environ.get("ENDPOINT_NAME", "compileml-scorer")
print(
    f"mode={MB_MODE}  "
    f"offline={SM_OFFLINE}  "
    f"image={IMAGE_URI}  "
    f"artifact={ARTIFACT.relative_to(ROOT)}"
)

# SMD support: launcher + staging live in sagemaker-ai/model_servers (smd_local.py)
sys.path.insert(0, str(ROOT / "sagemaker-ai" / "model_servers"))
# MODEL_SERVER=SMD IMAGE_URI=public.ecr.aws/sagemaker/sagemaker-distribution:3.2.0-cpu runs the
# real SageMaker Distribution image; its Tornado server needs code/inference.py (spec cell).
IS_SMD_IMAGE = "sagemaker-distribution" in IMAGE_URI

sagemaker.config INFO - Not applying SDK defaults from location: /Library/Application Support/sagemaker/config.yaml


sagemaker.config INFO - Not applying SDK defaults from location: /Users/denisburakov/Library/Application Support/sagemaker/config.yaml


mode=local  offline=True  image=compileml-scorer:local  artifact=artifacts/insurance_fraud.json


## Offline stubs

`sagemaker_offline.use_local_stubs()` swaps three SDK functions for local no-ops: `resolve_and_validate_role` (returns the ARN as given, like a caller without `iam:SimulatePrincipalPolicy`), `Session.default_bucket` (returns `"local"`), and `LocalContainerMode._pull_image` (connects to Docker, pulls nothing). The cloud path is untouched.

In [2]:
if SM_OFFLINE:
    from sagemaker_offline import use_local_stubs

    use_local_stubs()
    print("SDK local stubs applied: role validation, default bucket, image pull")

SDK local stubs applied: role validation, default bucket, image pull


## The model: an `InferenceSpec`

`InferenceSpec` is v3's contract for custom models: `load(model_dir)` and `invoke(input, model)` are what the SDK's own handler calls inside an AWS Deep Learning Container, and `prepare(model_dir)` runs on the host at build time. With a BYOC image the container runs our `serve.py`, so only `prepare()` has an effect here: it stages the artifact JSON into `model_path`, which becomes `/opt/ml/model` in the container (and `model.tar.gz` on a real endpoint).

In [3]:
class CompileMLSpec(InferenceSpec):
    """compileml artifact for ModelBuilder.

    With the BYOC image only prepare() matters.
    """

    def __init__(self, artifact_src: str) -> None:
        self.artifact_src = artifact_src
        self.artifact_name = Path(artifact_src).name

    def prepare(self, model_dir: str, *args, **kwargs) -> None:
        # TorchServe's local runner mounts <model_path> at /opt/ml/model, MMS's mounts
        # <model_path>/code. Stage the artifact in both so either MODEL_SERVER finds it.
        for d in (Path(model_dir), Path(model_dir) / "code"):
            d.mkdir(parents=True, exist_ok=True)
            shutil.copy2(self.artifact_src, d / self.artifact_name)

    # SDK-handler hooks: executed only inside a DLC with the SDK's generated inference.py
    def load(self, model_dir: str):

        from compileml.runtime import load_artifact

        return load_artifact(Path(model_dir) / self.artifact_name)

    def invoke(self, input_object, model):
        from compileml.runtime import decide

        names = model["features"]["names"]
        rows = (
            input_object["instances"]
            if isinstance(input_object, dict)
            else [input_object]
        )
        return {
            "predictions": [
                decide(model, [r.get(n) for n in names], top_k=3) for r in rows
            ]
        }


spec = CompileMLSpec(str(ARTIFACT))

if MODEL_SERVER == ModelServer.SMD:
    from smd_local import SMD_ENV, enable_smd_local_mode, stage_smd_code

    enable_smd_local_mode()  # sagemaker 3.21 has no local launcher for SMD
    if IS_SMD_IMAGE:

        class SMDSpec(CompileMLSpec):
            """InferenceSpec whose prepare() writes the handler the SMD image will import.

            This is the working SMD recipe on 3.21: the SDK's own SMD handler is only generated
            for CustomOrchestrator specs and cannot run on the 3.2.0 image (issue #6200). With a
            plain InferenceSpec nothing of the SDK's runs inside the container: prepare() stages
            code/inference.py (model_servers/smd_handler.py) plus a vendored compileml, and the
            image's Tornado server imports that. load()/invoke() are not used on this path either.
            """

            def prepare(self, model_dir: str, *args, **kwargs) -> None:
                """Prepare the SMD code for the model directory."""
                stage_smd_code(model_dir, self.artifact_src)

        spec = SMDSpec(str(ARTIFACT))
print("spec:", type(spec).__name__)
feature_names = json.loads(ARTIFACT.read_text())["features"]["names"]

# three rows from the cold-start file, in the request shape serve.py accepts
with INPUT_CSV.open(newline="") as fh:
    reader = csv.DictReader(fh)
    rows = [next(reader) for _ in range(3)]
instances = [
    {k: (float(r[k]) if r[k].strip() else None) for k in feature_names} for r in rows
]
sample_input = {"instances": instances, "explain": True, "top_k": 3}
sample_output = {
    "predictions": [{"band": "G01", "latent_int": 0, "pd": 0.05, "reasons": []}],
    "artifact_hash": "0" * 12,
}
print("features:", feature_names)

spec: CompileMLSpec
features: ['incident_severity', 'num_vehicles_involved', 'num_injuries', 'num_witnesses', 'police_report_available', 'injury_claim', 'vehicle_claim', 'incident_hour', 'customer_age', 'policy_deductable', 'policy_annual_premium', 'num_claims_past_year']


## The three model servers: TorchServe, MMS, SMD

`model_server` names a **serving toolkit contract**: which program owns the HTTP server inside the container, how it finds your code, and what shape your code has to take. With our own image the value never changes what runs; with an AWS image it decides everything.

| | TorchServe | MMS (Multi Model Server) | SMD (SageMaker Distribution) |
| --- | --- | --- | --- |
| what runs in the container | Java frontend (Netty) that owns the ports, batching and worker pool, plus Python worker processes | same architecture; the older project TorchServe was forked from | a Python Tornado server, single process |
| where it ships | PyTorch inference DLCs (`pytorch-inference:*`) | HuggingFace / MXNet DLCs, the SageMaker Inference Toolkit | the Studio image `sagemaker-distribution:*`, inference mode |
| what you write | `inference.py` with hook functions `model_fn`, `input_fn`, `predict_fn`, `output_fn` (the SDK generates one that calls your `InferenceSpec.load()` / `invoke()`) | same hooks, same generated file | one function, `handler(request)`, sync or async; `request.body` is bytes; return a dict for JSON (the SDK only generates it for `CustomOrchestrator`) |
| how the container finds it | `SAGEMAKER_PROGRAM=inference.py`, `SAGEMAKER_SUBMIT_DIRECTORY=/opt/ml/model/code` | same | `SAGEMAKER_INFERENCE_CODE=inference.handler`, `SAGEMAKER_INFERENCE_CODE_DIRECTORY=/opt/ml/model/code` |
| dependencies | `code/requirements.txt`, pip at start-up | same | `code/requirements.txt`, micromamba then pip |
| runs as | root | root | non-root `sagemaker-user` (staged files must be world-readable) |
| local runner in sagemaker 3.21 | `docker run <image> serve`, port 8080, mounts `model_path` at `/opt/ml/model` | same, but mounts **`model_path/code`**; `invoke()` wraps the response in a one-element list | **none**; `model_servers/smd_local.py` adds one |

**BYOC image.** Our `serve.py` is the HTTP server *and* the handler, so the toolkit inside the container is FastAPI on uvicorn whatever the value says. The SDK still stages the toolkit's scaffold (`code/inference.py`, `serve.pkl`, `requirements.txt`) and sets its env vars, and our entrypoint ignores both. Only the local runner's behaviour differs (the mount path and the list wrapping above), which is why `prepare()` stages the artifact in both places. On a real endpoint SageMaker runs `serve` on 8080 with `model.tar.gz` at `/opt/ml/model` for every value, so there the three are indistinguishable for our image.

**SMD image.** Its Tornado server does look for `inference.handler`, so to run the compileml artifact on it you supply that one function. That is `SMDSpec` below: an `InferenceSpec` whose `prepare()` stages `model_servers/smd_handler.py` as `code/inference.py` next to a vendored `compileml`. The SDK's own SMD recipe, `CustomOrchestrator`, is broken against the 3.2.0 image on SDK 3.21 (issue #6200: the image bundles a pre-CVE-2026-8596 SDK that still expects `SAGEMAKER_SERVE_SECRET_KEY`), see `model_servers/custom_orchestrator.py`.

**Rule of thumb.** BYOC image: `TORCHSERVE` (mounts `model_path` as-is, response unwrapped). SMD image: `SMD` with an `InferenceSpec` that writes the handler. DLC: the toolkit that DLC ships, with the SDK's generated handler calling your `load()`/`invoke()`.

## Part 1: ModelBuilder

- `model_server` is mandatory once `image_uri` is set, and `MODEL_SERVER=TORCHSERVE|MMS` (config cell) switches it. Both stage the same scaffold into `model_path` (`code/inference.py`, `code/serve.pkl` with the pickled spec and SchemaBuilder, `code/requirements.txt`; our entrypoint ignores it) and both launch `docker run <image> serve` on port 8080. What differs, measured (`model_server_matrix.json`):

  | | TORCHSERVE | MMS |
  | --- | --- | --- |
  | mounted at `/opt/ml/model` | `model_path` | `model_path/code` |
  | env the SDK sets | `SAGEMAKER_PROGRAM=""`, `SAGEMAKER_SUBMIT_DIRECTORY=""` (3.21 quirk) | `SAGEMAKER_PROGRAM=inference.py`, `SAGEMAKER_SUBMIT_DIRECTORY=/opt/ml/model/code` |
  | `LocalEndpoint.invoke().body` (a `BytesIO` of JSON) | the response object | the response wrapped in a one-element list |
  | on a real endpoint | identical: SageMaker runs `serve` with `model.tar.gz` at `/opt/ml/model` | identical |

  `prepare()` stages the artifact in both layouts and `as_json()` unwraps the list, so the rest of the notebook is the same for either. The other `ModelServer` values either reject this sample input at build (Triton, DJL, TGI, TensorFlow Serving), ask Docker for a GPU (TEI), or have no local launcher at all in 3.21 (SMD, vLLM, SGLang, llama.cpp). SMD works here too: `MODEL_SERVER=SMD` uses the launcher from `model_servers/smd_local.py`, and with `IMAGE_URI=public.ecr.aws/sagemaker/sagemaker-distribution:3.2.0-cpu` the spec cell switches to `SMDSpec`, whose `prepare()` stages `code/inference.py` (`model_servers/smd_handler.py`) for the image's Tornado server. That plain-`InferenceSpec` route is the working SMD recipe on 3.21; the SDK's `CustomOrchestrator` route is broken against this image (issue #6200, see `model_servers/custom_orchestrator.py`).
- `model_path` is a fresh temp directory; `prepare()` drops the artifact into it and ModelBuilder adds `code/`.
- `dependencies={"auto": False}` skips the pickle-based dependency detector.
- `SchemaBuilder` picks the client-side serializer and deserializer for `invoke()` from the samples (dict samples: JSON).

In [4]:
builder = ModelBuilder(
    inference_spec=spec,
    model_server=MODEL_SERVER,
    schema_builder=SchemaBuilder(
        sample_input=sample_input, sample_output=sample_output
    ),
    image_uri=IMAGE_URI,
    mode=Mode.LOCAL_CONTAINER if MB_MODE == "local" else Mode.SAGEMAKER_ENDPOINT,
    model_path=tempfile.mkdtemp(prefix="compileml-mb-"),
    role_arn=ROLE_ARN,
    dependencies={"auto": False},
    env_vars={"ARTIFACT_PATH": f"/opt/ml/model/{ARTIFACT.name}"},
)
builder.build()
model_path = Path(builder.model_path)
mount = {
    ModelServer.TORCHSERVE: model_path,
    ModelServer.MMS: model_path / "code",
    ModelServer.SMD: model_path,
}[MODEL_SERVER]
print(
    f"model_server={MODEL_SERVER.name}: local runner will mount {mount.name if mount != model_path else 'model_path'} at /opt/ml/model"
)
print("staged model_path:", model_path)
for p in sorted(model_path.rglob("*")):
    print(
        "   ",
        p.relative_to(model_path),
        f"({p.stat().st_size} B)" if p.is_file() else "",
    )

if MODEL_SERVER == ModelServer.SMD and IS_SMD_IMAGE:
    # dependencies={"auto": False} leaves an empty code/requirements.txt; the SMD image would run
    # `micromamba install --file` on it, log an [ERROR] on failure, and the SDK aborts on that line.
    req = model_path / "code" / "requirements.txt"
    if req.exists() and req.stat().st_size == 0:
        req.unlink()


[09/16/26 08:04:33] DEBUG    Auto-detecting optimal instance type for model...           ]8;id=5338015;file:///Users/denisburakov/Documents/Git-deburky/compileml-fraud-scoring/.venv/lib/python3.12/site-packages/sagemaker/serve/model_builder_utils.py\model_builder_utils.py]8;;\:]8;id=5338016;file:///Users/denisburakov/Documents/Git-deburky/compileml-fraud-scoring/.venv/lib/python3.12/site-packages/sagemaker/serve/model_builder_utils.py#341\341]8;;\

                    DEBUG    Using default CPU instance type: ml.m5.large                ]8;id=5338022;file:///Users/denisburakov/Documents/Git-deburky/compileml-fraud-scoring/.venv/lib/python3.12/site-packages/sagemaker/serve/model_builder_utils.py\model_builder_utils.py]8;;\:]8;id=5338023;file:///Users/denisburakov/Documents/Git-deburky/compileml-fraud-scoring/.venv/lib/python3.12/site-packages/sagemaker/serve/model_builder_utils.py#375\375]8;;\

                    INFO     SageMaker Python SDK will collect telemetry to help us better ]8;id=5338030;file:///Users/denisburakov/Documents/Git-deburky/compileml-fraud-scoring/.venv/lib/python3.12/site-packages/sagemaker/core/telemetry/telemetry_logging.py\telemetry_logging.py]8;;\:]8;id=5338031;file:///Users/denisburakov/Documents/Git-deburky/compileml-fraud-scoring/.venv/lib/python3.12/site-packages/sagemaker/core/telemetry/telemetry_logging.py#322\322]8;;\
                             understand our user's needs, diagnose issues, and deliver                             
                             additional features.                                                                  
                             To opt out of telemetry, please disable via TelemetryOptOut                           
                             parameter in SDK defaults config. For more information, refer                         
                             to                                                                                    
                             https://sagemaker.readthedocs.io/en/stable/overview.html#conf                         
                             iguring-and-using-defaults-with-the-sagemaker-python-sdk.                             

                    DEBUG    Either inference spec or model is provided. ModelBuilder   ]8;id=5338037;file:///Users/denisburakov/Documents/Git-deburky/compileml-fraud-scoring/.venv/lib/python3.12/site-packages/sagemaker/serve/model_builder_utils.py\model_builder_utils.py]8;;\:]8;id=5338038;file:///Users/denisburakov/Documents/Git-deburky/compileml-fraud-scoring/.venv/lib/python3.12/site-packages/sagemaker/serve/model_builder_utils.py#1380\1380]8;;\
                             is not handling MLflow model input                                                    

                    DEBUG    Skipping auto-detection as image_uri is provided:           ]8;id=5338044;file:///Users/denisburakov/Documents/Git-deburky/compileml-fraud-scoring/.venv/lib/python3.12/site-packages/sagemaker/serve/model_builder_utils.py\model_builder_utils.py]8;;\:]8;id=5338045;file:///Users/denisburakov/Documents/Git-deburky/compileml-fraud-scoring/.venv/lib/python3.12/site-packages/sagemaker/serve/model_builder_utils.py#899\899]8;;\
                             compileml-scorer:local                                                                

                    INFO     Found credentials in shared credentials file: ~/.aws/credentials   ]8;id=5338052;file:///Users/denisburakov/Documents/Git-deburky/compileml-fraud-scoring/.venv/lib/python3.12/site-packages/botocore/credentials.py\credentials.py]8;;\:]8;id=5338053;file:///Users/denisburakov/Documents/Git-deburky/compileml-fraud-scoring/.venv/lib/python3.12/site-packages/botocore/credentials.py#1392\1392]8;;\

                    INFO     ✅ Model has been created: 'model-24774e4f' using server         ]8;id=5338060;file:///Users/denisburakov/Documents/Git-deburky/compileml-fraud-scoring/.venv/lib/python3.12/site-packages/sagemaker/serve/model_builder.py\model_builder.py]8;;\:]8;id=5338061;file:///Users/denisburakov/Documents/Git-deburky/compileml-fraud-scoring/.venv/lib/python3.12/site-packages/sagemaker/serve/model_builder.py#4429\4429]8;;\
                             TORCHSERVE in LOCAL_CONTAINER mode                                                    

model_server=TORCHSERVE: local runner will mount model_path at /opt/ml/model
staged model_path: /var/folders/03/rlvlg_197_n_4hq35cfshvpw0000gn/T/compileml-mb-uetggdtd
    code 
    code/inference.py (7279 B)
    code/insurance_fraud.json (45741 B)
    code/metadata.json (176 B)
    code/requirements.txt (0 B)
    code/serve.pkl (4233 B)
    insurance_fraud.json (45741 B)
    shared_libs 


## Deploy locally

`deploy_local()` starts the image (no pull with the stubs) with `model_path` mounted at `/opt/ml/model`, polls `GET /ping`, and returns a `LocalEndpoint`. In sagemaker 3.21 `deploy()` in local mode returns the same `LocalEndpoint`, not a `Predictor`, so there is no `.predict()`; scoring goes through `invoke()`.

You will see a `RuntimeError: Failed to run: ['docker', 'compose', ...]` traceback from a thread here. `LocalEndpoint.create()` starts the container with `docker run` (that one serves you), then registers the endpoint with the old local-mode session, whose `create_endpoint` starts it a second time through docker compose on the same port. Docker refuses the port, compose exits 1, the SDK's log thread raises, and `deploy_local` carries on with container 1. Harmless; the teardown cell removes the dead second container.

The cell can be re-run: it first drops the endpoint name from the local session's registry, otherwise the SDK would hand back a stale handle without a container.

In [5]:
assert MB_MODE == "local", (
    "set MB_MODE=local for this section; the endpoint path is at the end"
)
# Re-running this cell in the same kernel: the local session keeps endpoint names in a
# class-level registry that delete() never clears, and deploy_local() then returns a bare
# handle with no container ("Model server or container mode not available"). Clear it first.
from sagemaker.core.local.local_session import LocalSagemakerClient

for registry in (
    LocalSagemakerClient._endpoints,
    LocalSagemakerClient._endpoint_configs,
):
    registry.pop(ENDPOINT_NAME, None)

local_endpoint = builder.deploy_local(
    endpoint_name=ENDPOINT_NAME, container_timeout_in_seconds=300
)
print(type(local_endpoint).__name__, local_endpoint.endpoint_name)

                    INFO     Found credentials in shared credentials file: ~/.aws/credentials   ]8;id=5338066;file:///Users/denisburakov/Documents/Git-deburky/compileml-fraud-scoring/.venv/lib/python3.12/site-packages/botocore/credentials.py\credentials.py]8;;\:]8;id=5338067;file:///Users/denisburakov/Documents/Git-deburky/compileml-fraud-scoring/.venv/lib/python3.12/site-packages/botocore/credentials.py#1392\1392]8;;\

                    INFO     Waiting for model server TORCHSERVE to start up...         ]8;id=5338074;file:///Users/denisburakov/Documents/Git-deburky/compileml-fraud-scoring/.venv/lib/python3.12/site-packages/sagemaker/serve/mode/local_container_mode.py\local_container_mode.py]8;;\:]8;id=5338075;file:///Users/denisburakov/Documents/Git-deburky/compileml-fraud-scoring/.venv/lib/python3.12/site-packages/sagemaker/serve/mode/local_container_mode.py#135\135]8;;\

[09/16/26 08:04:59] INFO     Pinging local endpoint...                                       ]8;id=5338082;file:///Users/denisburakov/Documents/Git-deburky/compileml-fraud-scoring/.venv/lib/python3.12/site-packages/sagemaker/serve/local_resources.py\local_resources.py]8;;\:]8;id=5338083;file:///Users/denisburakov/Documents/Git-deburky/compileml-fraud-scoring/.venv/lib/python3.12/site-packages/sagemaker/serve/local_resources.py#128\128]8;;\

                    INFO     'Docker Compose' found using Docker CLI.                                  ]8;id=5338090;file:///Users/denisburakov/Documents/Git-deburky/compileml-fraud-scoring/.venv/lib/python3.12/site-packages/sagemaker/core/local/image.py\image.py]8;;\:]8;id=5338091;file:///Users/denisburakov/Documents/Git-deburky/compileml-fraud-scoring/.venv/lib/python3.12/site-packages/sagemaker/core/local/image.py#168\168]8;;\

                    INFO     serving                                                                   ]8;id=5338097;file:///Users/denisburakov/Documents/Git-deburky/compileml-fraud-scoring/.venv/lib/python3.12/site-packages/sagemaker/core/local/image.py\image.py]8;;\:]8;id=5338098;file:///Users/denisburakov/Documents/Git-deburky/compileml-fraud-scoring/.venv/lib/python3.12/site-packages/sagemaker/core/local/image.py#338\338]8;;\

...

                    INFO     Using the long-lived AWS credentials found in session                    ]8;id=5338111;file:///Users/denisburakov/Documents/Git-deburky/compileml-fraud-scoring/.venv/lib/python3.12/site-packages/sagemaker/core/local/image.py\image.py]8;;\:]8;id=5338112;file:///Users/denisburakov/Documents/Git-deburky/compileml-fraud-scoring/.venv/lib/python3.12/site-packages/sagemaker/core/local/image.py#1132\1132]8;;\

                    INFO     docker compose file:                                                      ]8;id=5338118;file:///Users/denisburakov/Documents/Git-deburky/compileml-fraud-scoring/.venv/lib/python3.12/site-packages/sagemaker/core/local/image.py\image.py]8;;\:]8;id=5338119;file:///Users/denisburakov/Documents/Git-deburky/compileml-fraud-scoring/.venv/lib/python3.12/site-packages/sagemaker/core/local/image.py#787\787]8;;\
                             networks:                                                                             
                               sagemaker-local:                                                                    
                                 name: sagemaker-local                                                             
                             services:                                                                             
                               algo-1-4omg1:                                                                       
                                 command: serve                                                                    
                                 container_name: 4qvu6otc9l-algo-1-4omg1                                           
                                 environment:                                                                      
                                 - '[Masked]'                                                                      
                                 - '[Masked]'                                                                      
                                 - '[Masked]'                                                                      
                                 - '[Masked]'                                                                      
                                 - '[Masked]'                                                                      
                                 - '[Masked]'                                                                      
                                 - '[Masked]'                                                                      
                                 image: compileml-scorer:local                                                     
                                 networks:                                                                         
                                   sagemaker-local:                                                                
                                     aliases:                                                                      
                                     - algo-1-4omg1                                                                
                                 ports:                                                                            
                                 - 8080:8080                                                                       
                                 stdin_open: true                                                                  
                                 tty: true                                                                         
                                 volumes:                                                                          
                                 -                                                                                 
                             /var/folders/03/rlvlg_197_n_4hq35cfshvpw0000gn/T/compileml-mb-uetggdtd:/o             
                             pt/ml/model                                                                           
                             version: '2.3'                                                                        
                                                                                                                   

                    INFO     docker command: docker compose -f                                         ]8;id=5338125;file:///Users/denisburakov/Documents/Git-deburky/compileml-fraud-scoring/.venv/lib/python3.12/site-packages/sagemaker/core/local/image.py\image.py]8;;\:]8;id=5338126;file:///Users/denisburakov/Documents/Git-deburky/compileml-fraud-scoring/.venv/lib/python3.12/site-packages/sagemaker/core/local/image.py#811\811]8;;\
                             /private/var/folders/03/rlvlg_197_n_4hq35cfshvpw0000gn/T/tmp8c82b6lw/dock             
                             er-compose.yaml up --build --abort-on-container-exit                                  

                    INFO     Checking if serving container is up, attempt: 5                        ]8;id=5338133;file:///Users/denisburakov/Documents/Git-deburky/compileml-fraud-scoring/.venv/lib/python3.12/site-packages/sagemaker/core/local/entities.py\entities.py]8;;\:]8;id=5338134;file:///Users/denisburakov/Documents/Git-deburky/compileml-fraud-scoring/.venv/lib/python3.12/site-packages/sagemaker/core/local/entities.py#671\671]8;;\

LocalEndpoint compileml-scorer


## Invoke 1: `LocalEndpoint.invoke()`

The SDK serializes `body` with the SchemaBuilder's serializer (JSON), posts to `http://localhost:8080/invocations`, and returns an `InvokeEndpointOutput` whose `body` is whatever the deserializer produced. In 3.21 with a dict `sample_output` that is a `BytesIO` of the raw JSON, so it is read and parsed here.

In [6]:
def as_json(body):
    """Read as JSON. LocalEndpoint.invoke() returns a BytesIO of json.dumps(deserialized);
    on the MMS path the deserialized value is a one-element list, so unwrap after parsing."""
    if hasattr(body, "read"):
        body = body.read()
    if isinstance(body, (bytes, bytearray)):
        body = body.decode()
    if isinstance(body, str):
        body = json.loads(body)
    if isinstance(body, list) and len(body) == 1 and isinstance(body[0], dict):
        body = body[0]
    return body


out = local_endpoint.invoke(body=sample_input)
result_local = as_json(out.body)
for i, p in enumerate(result_local["predictions"]):
    print(i, p["band"], p["latent_int"], p["pd"], [r["code"] for r in p["reasons"]])
print("artifact_hash:", result_local["artifact_hash"])

0 G06 1000 1.0 ['VEHICLES', 'VEHICLE_AMOUNT', 'INJURY_AMOUNT']
1 G06 715 1.0 ['INJURIES', 'SEVERITY', 'VEHICLE_AMOUNT']
2 G06 212 0.212941 ['INJURY_AMOUNT', 'PREMIUM', 'VEHICLE_AMOUNT']
artifact_hash: 8a9bae69bb72


## Invoke 2: `invoke_endpoint()`, the boto3 approach, local

Against a real endpoint an application calls `boto3.client("sagemaker-runtime").invoke_endpoint(EndpointName=..., ContentType=..., Body=...)`. Local mode ships `LocalSagemakerRuntimeClient` with the same `invoke_endpoint()` signature, pointed at `http://localhost:8080/invocations`, so the boto3-shaped call runs unchanged against Part 1's container. The response is a dict with `Body` (a stream to `.read()`) and `ContentType`, like boto3's.

`sagemaker.core.resources.Endpoint.invoke()` is the v3 resource wrapper around exactly that call; it fetches its client from the `SageMakerClient` singleton, so swapping the local runtime client in there routes it to the same container. Both are run and compared.

`Endpoint` is sagemaker-core's pydantic mirror of the API resource, and its methods map one-to-one onto API calls:

| call | API | local mode |
| --- | --- | --- |
| `Endpoint(endpoint_name=...)` | none: builds a handle, no request | fine |
| `Endpoint.get(name)` | DescribeEndpoint | needs the local control-plane client (Part 2) |
| `Endpoint.create(...)` | CreateEndpoint (control plane) | needs the local control-plane client (Part 2); `ModelBuilder.deploy_local()` → `LocalEndpoint` is the stand-in in Part 1 |
| `endpoint.invoke(body=...)` | runtime InvokeEndpoint | works once the runtime client is the local one |

There is no `invoke_endpoint()` on the resource in 3.21; that name is the runtime client's method, which is what is called first above.

This is where this gets interesting:

```python
from sagemaker.core.resources import Endpoint, Model
from sagemaker.core.shapes import EndpointConfig

# Create a model resource
model = Model.create(
    model_name="my-model",
    primary_container={
        "image": "your-inference-image",
        "model_data_url": "s3://your-bucket/model.tar.gz"
    },
    execution_role_arn="your-sagemaker-role"
)

# Deploy to an endpoint
endpoint = Endpoint.create(
    endpoint_name="my-endpoint",
    endpoint_config_name="my-config",
    model_name=model.model_name
)

# Make predictions
response = endpoint.invoke_endpoint(
    body=b'{"instances": [1, 2, 3, 4]}',
    content_type="application/json"
)
```

That snippet is from the sagemaker-core [**README**](https://sagemaker.readthedocs.io/en/v3docs/sagemaker_core/index.html) and does not match the installed SDK (3.21): `Endpoint.create()` has no `model_name` parameter (it takes `endpoint_name`, `endpoint_config_name`, `deployment_config`, `tags`), the resource has `invoke()` but no `invoke_endpoint()`, and the `EndpointConfig` it imports is never used. The real chain is the API's: **Model → EndpointConfig → Endpoint**, shown working locally in Part 2 below. Here in Invoke 2 the endpoint already exists (Part 1 deployed it), so a handle is enough.

In [7]:
from sagemaker.core.local.local_session import LocalSagemakerRuntimeClient
from sagemaker.core.resources import Endpoint
from sagemaker.core.utils.utils import SageMakerClient

# The boto3 approach, local: LocalSagemakerRuntimeClient has the same invoke_endpoint()
# signature as boto3.client("sagemaker-runtime") and posts to http://localhost:8080/invocations.
runtime = LocalSagemakerRuntimeClient()
resp = runtime.invoke_endpoint(
    EndpointName=ENDPOINT_NAME,
    ContentType="application/json",
    Accept="application/json",
    Body=json.dumps(sample_input),
)
result_core = json.loads(resp["Body"].read())
print(
    "invoke_endpoint ->",
    resp["ContentType"],
    "|",
    result_core["predictions"][0]["band"],
    result_core["predictions"][0]["pd"],
)

# The sagemaker.core resource wrapper makes exactly that call (resources.py line 10765). It takes
# its client from the SageMakerClient singleton, so point the singleton at the local runtime
# client for the duration of the call and Endpoint(name).invoke() hits the same container.
clients = SageMakerClient()
real_runtime_client = clients.sagemaker_runtime_client
clients.sagemaker_runtime_client = runtime
try:
    endpoint = Endpoint(
        endpoint_name=ENDPOINT_NAME
    )  # handle, no API call; Part 1's container answers
    resp2 = endpoint.invoke(
        body=json.dumps(sample_input),
        content_type="application/json",
        accept="application/json",
    )
    result_resource = json.loads(resp2.body.read())
finally:
    clients.sagemaker_runtime_client = real_runtime_client  # put boto3 back

print(json.dumps(result_core["predictions"][0], indent=2))
assert result_core == result_resource == result_local, (
    "all invoke paths must return the same decisions"
)
print("runtime.invoke_endpoint() == Endpoint.invoke() == LocalEndpoint.invoke():", True)

Attaching to 4qvu6otc9l-algo-1-4omg1
invoke_endpoint -> application/json | G06 1.0


                    DEBUG    No boto3 session provided. Creating a new session.                        ]8;id=5338141;file:///Users/denisburakov/Documents/Git-deburky/compileml-fraud-scoring/.venv/lib/python3.12/site-packages/sagemaker/core/utils/utils.py\utils.py]8;;\:]8;id=5338142;file:///Users/denisburakov/Documents/Git-deburky/compileml-fraud-scoring/.venv/lib/python3.12/site-packages/sagemaker/core/utils/utils.py#357\357]8;;\

                    WARNING  No region provided. Using default region.                                 ]8;id=5338148;file:///Users/denisburakov/Documents/Git-deburky/compileml-fraud-scoring/.venv/lib/python3.12/site-packages/sagemaker/core/utils/utils.py\utils.py]8;;\:]8;id=5338149;file:///Users/denisburakov/Documents/Git-deburky/compileml-fraud-scoring/.venv/lib/python3.12/site-packages/sagemaker/core/utils/utils.py#361\361]8;;\

                    DEBUG    No config provided. Using default config.                                 ]8;id=5338155;file:///Users/denisburakov/Documents/Git-deburky/compileml-fraud-scoring/.venv/lib/python3.12/site-packages/sagemaker/core/utils/utils.py\utils.py]8;;\:]8;id=5338156;file:///Users/denisburakov/Documents/Git-deburky/compileml-fraud-scoring/.venv/lib/python3.12/site-packages/sagemaker/core/utils/utils.py#365\365]8;;\

                    INFO     Found credentials in shared credentials file: ~/.aws/credentials   ]8;id=5338161;file:///Users/denisburakov/Documents/Git-deburky/compileml-fraud-scoring/.venv/lib/python3.12/site-packages/botocore/credentials.py\credentials.py]8;;\:]8;id=5338162;file:///Users/denisburakov/Documents/Git-deburky/compileml-fraud-scoring/.venv/lib/python3.12/site-packages/botocore/credentials.py#1392\1392]8;;\

{
  "band": "G06",
  "latent_int": 1000,
  "pd": 1.0,
  "reasons": [
    {
      "code": "VEHICLES",
      "impact_int": 715,
      "message": "Unusual number of vehicles involved."
    },
    {
      "code": "VEHICLE_AMOUNT",
      "impact_int": 174,
      "message": "Vehicle claim amount is high."
    },
    {
      "code": "INJURY_AMOUNT",
      "impact_int": 134,
      "message": "Injury claim amount is high."
    }
  ]
}
runtime.invoke_endpoint() == Endpoint.invoke() == LocalEndpoint.invoke(): True


In [8]:
SAGEMAKER_ENDPOINT_NAME = ENDPOINT_NAME
print(f"Endpoint name: {SAGEMAKER_ENDPOINT_NAME}")

%store SAGEMAKER_ENDPOINT_NAME

Endpoint name: compileml-scorer
Stored 'SAGEMAKER_ENDPOINT_NAME' (str)


In [9]:
!echo $SAGEMAKER_ENDPOINT_NAME

compileml-scorer


## Tear down the local container

`LocalEndpoint.delete()` stops the SDK's local endpoint but in 3.21 leaves the containers it started behind (one from its direct `docker run`, one from its docker compose pass). The next deploy then fails with `port 8080 already allocated`, so the leftovers are removed explicitly. `delete()` can also raise `ProcessLookupError` when the compose process it signals has already exited; that is harmless.

In [10]:
try:
    local_endpoint.delete()
except (
    ProcessLookupError
) as exc:  # the SDK signals its compose process, which may already have exited
    print("delete():", exc)

import docker

client = docker.from_env()


def ours(
    c,
):  # our image, or anything the SDK left on port 8080 / named algo-1 (older builds of the tag)
    ports = c.attrs.get("HostConfig", {}).get("PortBindings") or {}
    return (
        IMAGE_URI in (c.image.tags or []) or "8080/tcp" in ports or "algo-1" in c.name
    )


for c in client.containers.list(all=True):
    if ours(c):
        print("removing", c.name, c.status)
        c.remove(force=True)
print("remaining:", [c.name for c in client.containers.list(all=True) if ours(c)])

removing 4qvu6otc9l-algo-1-4omg1 created
removing funny_mendeleev running


remaining: []


## Part 2: the same endpoint through `sagemaker.core` resources only, still local

No `ModelBuilder` this time. `Model.create()`, `EndpointConfig.create()` and `Endpoint.create()` are the control-plane calls an application makes for a real endpoint, and each one fetches its client from the `SageMakerClient` singleton. Local mode ships `LocalSagemakerClient`, which implements `create_model`, `create_endpoint_config`, `create_endpoint`, `describe_*` and `delete_*` by keeping the objects in memory and starting the container through docker compose. Swap it into the singleton next to the local runtime client and the identical `Endpoint.create()` code runs on this machine.

Two more things to know:

- `sagemaker.core.local.image` references `sagemaker.serve.model_builder` without importing it, so that module is imported up front (the same workaround `local_transform.py` uses).
- Scoring uses `invoke_endpoint(EndpointName=..., ContentType=..., Body=...)` on the runtime client, boto3-shaped: on AWS that client is `boto3.client("sagemaker-runtime")`, here it is `LocalSagemakerRuntimeClient`. `Endpoint.invoke()` is the resource wrapper around that same call (resources.py line 10765) and is run once alongside to show they agree.

In [11]:
# Part 2 needs sagemaker.serve.model_builder loaded: core/local/image.py reads
# sagemaker.serve.model_builder.DIR_PARAM_NAME without importing it. The ModelBuilder
# import in the first cell already did that for this kernel.
from sagemaker.core.local import LocalSession
from sagemaker.core.local.local_session import LocalSagemakerClient
from sagemaker.core.resources import Endpoint, EndpointConfig, Model
from sagemaker.core.shapes import ContainerDefinition, ProductionVariant
from sagemaker.core.utils.utils import SageMakerClient

CORE_NAME = "compileml-core-local"
core_model_path = Path(tempfile.mkdtemp(prefix="compileml-core-"))
shutil.copy2(
    ARTIFACT, core_model_path / ARTIFACT.name
)  # unpacked model dir; a model.tar.gz works too

local = LocalSession()
local.config = {"local": {"local_code": True}}
clients = SageMakerClient()  # singleton used by every sagemaker.core resource
saved_clients = (clients.sagemaker_client, clients.sagemaker_runtime_client)
clients.sagemaker_client = local.sagemaker_client  # CreateModel/CreateEndpointConfig/CreateEndpoint/Describe*/Delete* -> local registry + docker compose
clients.sagemaker_runtime_client = (
    local.sagemaker_runtime_client
)  # InvokeEndpoint -> http://localhost:8080/invocations
print(
    "control plane:",
    type(clients.sagemaker_client).__name__,
    " data plane:",
    type(clients.sagemaker_runtime_client).__name__,
)

core_env = {"ARTIFACT_PATH": f"/opt/ml/model/{ARTIFACT.name}"}
if IS_SMD_IMAGE:  # the SMD image needs the handler staged and told where it is
    from smd_local import SMD_ENV, stage_smd_code

    stage_smd_code(core_model_path, ARTIFACT)
    core_env.update(SMD_ENV)


                    INFO     Found credentials in shared credentials file: ~/.aws/credentials   ]8;id=5338167;file:///Users/denisburakov/Documents/Git-deburky/compileml-fraud-scoring/.venv/lib/python3.12/site-packages/botocore/credentials.py\credentials.py]8;;\:]8;id=5338168;file:///Users/denisburakov/Documents/Git-deburky/compileml-fraud-scoring/.venv/lib/python3.12/site-packages/botocore/credentials.py#1392\1392]8;;\

control plane: LocalSagemakerClient  data plane: LocalSagemakerRuntimeClient


In [12]:
for registry in (
    LocalSagemakerClient._endpoints,
    LocalSagemakerClient._endpoint_configs,
    LocalSagemakerClient._models,
):
    registry.pop(CORE_NAME, None)  # re-runnable in one kernel, as above
try:
    core_model = Model.create(
        model_name=CORE_NAME,
        primary_container=ContainerDefinition(
            image=IMAGE_URI,
            model_data_url=f"file://{core_model_path}",
            environment=core_env,
        ),
        execution_role_arn=ROLE_ARN,  # stored, never validated locally
    )
    core_config = EndpointConfig.create(
        endpoint_config_name=CORE_NAME,
        production_variants=[
            ProductionVariant(
                variant_name="AllTraffic",
                model_name=CORE_NAME,
                initial_instance_count=1,
                instance_type="local",
            )
        ],
    )
    core_endpoint = Endpoint.create(
        endpoint_name=CORE_NAME, endpoint_config_name=CORE_NAME
    )  # starts the container, waits for /ping
    print(f"Creating endpoint: {CORE_NAME}")
    core_endpoint.wait_for_status(
        target_status="InService"
    )  # polls describe_endpoint, local client answers
    print(f"Endpoint ready: {CORE_NAME}", core_endpoint.endpoint_status)

    # score through the runtime client, boto3-shaped: the same call deploy_modelbuilder.py makes
    # with boto3.client("sagemaker-runtime") against a cloud endpoint. Locally the client is
    # LocalSagemakerRuntimeClient, whose invoke_endpoint posts to http://localhost:8080/invocations.
    runtime = clients.sagemaker_runtime_client
    resp = runtime.invoke_endpoint(
        EndpointName=CORE_NAME,
        ContentType="application/json",
        Accept="application/json",
        Body=json.dumps(sample_input),
    )
    result_create = json.loads(resp["Body"].read())

    # the resource wrapper does exactly that call (resources.py line 10765: client.invoke_endpoint(...))
    result_boto_shaped = json.loads(
        core_endpoint.invoke(
            body=json.dumps(sample_input),
            content_type="application/json",
            accept="application/json",
        ).body.read()
    )
finally:
    pass  # clients are restored after teardown below

for i, p in enumerate(result_create["predictions"]):
    print(i, p["band"], p["latent_int"], p["pd"], [r["code"] for r in p["reasons"]])
assert result_create == result_boto_shaped, (
    "Endpoint.invoke() must equal the invoke_endpoint() it wraps"
)
if "result_local" in globals():
    assert result_create == result_local, (
        "Part 2 must return the same decisions as Part 1"
    )
    print(
        "Endpoint.create() + runtime.invoke_endpoint() == Endpoint.invoke() == LocalEndpoint.invoke():",
        True,
    )

sagemaker.config INFO - Not applying SDK defaults from location: /Library/Application Support/sagemaker/config.yaml


sagemaker.config INFO - Not applying SDK defaults from location: /Users/denisburakov/Library/Application Support/sagemaker/config.yaml


[09/16/26 08:05:00] INFO     Creating model resource.                                            ]8;id=5338175;file:///Users/denisburakov/Documents/Git-deburky/compileml-fraud-scoring/.venv/lib/python3.12/site-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=5338176;file:///Users/denisburakov/Documents/Git-deburky/compileml-fraud-scoring/.venv/lib/python3.12/site-packages/sagemaker/core/resources.py#20592\20592]8;;\

                    INFO     Creating endpoint_config resource.                                  ]8;id=5338182;file:///Users/denisburakov/Documents/Git-deburky/compileml-fraud-scoring/.venv/lib/python3.12/site-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=5338183;file:///Users/denisburakov/Documents/Git-deburky/compileml-fraud-scoring/.venv/lib/python3.12/site-packages/sagemaker/core/resources.py#11068\11068]8;;\

                    INFO     Creating endpoint resource.                                         ]8;id=5338189;file:///Users/denisburakov/Documents/Git-deburky/compileml-fraud-scoring/.venv/lib/python3.12/site-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=5338190;file:///Users/denisburakov/Documents/Git-deburky/compileml-fraud-scoring/.venv/lib/python3.12/site-packages/sagemaker/core/resources.py#10227\10227]8;;\

...

[09/16/26 08:05:05] INFO     Checking if serving container is up, attempt: 10                       ]8;id=5338264;file:///Users/denisburakov/Documents/Git-deburky/compileml-fraud-scoring/.venv/lib/python3.12/site-packages/sagemaker/core/local/entities.py\entities.py]8;;\:]8;id=5338265;file:///Users/denisburakov/Documents/Git-deburky/compileml-fraud-scoring/.venv/lib/python3.12/site-packages/sagemaker/core/local/entities.py#671\671]8;;\

Output()

                    INFO     Final Resource Status: InService                                    ]8;id=5338271;file:///Users/denisburakov/Documents/Git-deburky/compileml-fraud-scoring/.venv/lib/python3.12/site-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=5338272;file:///Users/denisburakov/Documents/Git-deburky/compileml-fraud-scoring/.venv/lib/python3.12/site-packages/sagemaker/core/resources.py#10483\10483]8;;\

5743bkle9e-algo-1-vu0l8  | INFO:     192.168.65.1:48662 - "GET /ping HTTP/1.1" 200 OK
Creating endpoint: compileml-core-local


Endpoint ready: compileml-core-local InService
5743bkle9e-algo-1-vu0l8  | INFO:     192.168.65.1:19650 - "POST /invocations HTTP/1.1" 200 OK
0 G06 1000 1.0 ['VEHICLES', 'VEHICLE_AMOUNT', 'INJURY_AMOUNT']
1 G06 715 1.0 ['INJURIES', 'SEVERITY', 'VEHICLE_AMOUNT']
2 G06 212 0.212941 ['INJURY_AMOUNT', 'PREMIUM', 'VEHICLE_AMOUNT']
Endpoint.create() + runtime.invoke_endpoint() == Endpoint.invoke() == LocalEndpoint.invoke(): True
5743bkle9e-algo-1-vu0l8  | INFO:     192.168.65.1:19650 - "POST /invocations HTTP/1.1" 200 OK


### Tear down (local `delete_endpoint` stops the compose container)

In [13]:
import docker

client = docker.from_env()

try:
    core_endpoint.delete()
    core_config.delete()
    core_model.delete()
finally:
    clients.sagemaker_client, clients.sagemaker_runtime_client = (
        saved_clients  # boto3 back
    )

for c in client.containers.list(all=True):
    if ours(c):
        print("removing", c.name, c.status)
        c.remove(force=True)
print("remaining:", [c.name for c in client.containers.list(all=True) if ours(c)])

                    INFO     Deleting Endpoint - compileml-core-local                            ]8;id=5338286;file:///Users/denisburakov/Documents/Git-deburky/compileml-fraud-scoring/.venv/lib/python3.12/site-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=5338287;file:///Users/denisburakov/Documents/Git-deburky/compileml-fraud-scoring/.venv/lib/python3.12/site-packages/sagemaker/core/resources.py#10427\10427]8;;\

                    INFO     Deleting EndpointConfig - compileml-core-local                      ]8;id=5338293;file:///Users/denisburakov/Documents/Git-deburky/compileml-fraud-scoring/.venv/lib/python3.12/site-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=5338294;file:///Users/denisburakov/Documents/Git-deburky/compileml-fraud-scoring/.venv/lib/python3.12/site-packages/sagemaker/core/resources.py#11219\11219]8;;\

                    INFO     Deleting Model - compileml-core-local                               ]8;id=5338300;file:///Users/denisburakov/Documents/Git-deburky/compileml-fraud-scoring/.venv/lib/python3.12/site-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=5338301;file:///Users/denisburakov/Documents/Git-deburky/compileml-fraud-scoring/.venv/lib/python3.12/site-packages/sagemaker/core/resources.py#20739\20739]8;;\

removing 5743bkle9e-algo-1-vu0l8 exited
remaining: []
